# Dynamics 05 (Rust) — Agents-programmes à budget explicite : le compagnon Rust du module Lean

**Navigation** : [<< Dynamics-04-Open-Source-Game-Theory-Python](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) | [Version Python](Dynamics-05-Agents-Bornes-Python.ipynb) | [Version C#](Dynamics-05-Agents-Bornes-CSharp.ipynb) | [Index](README.md) | [companion Lean](Dynamics-05-Agents-Bornes-Lean.ipynb) | [Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python >>](Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python.ipynb)

Ce notebook est le compagnon **Rust** du module Lean 4 [`ProgramGames.Bounded`](../game_theory_lean/ProgramGames/Bounded.lean) du lake [`game_theory_lean`](../game_theory_lean/README.md). Le module represente explicitement le **code public** et le **budget de raisonnement fini** d'un agent-programme (modele structurel de Barasz et al. 2014 et Critch 2016), avec un interprete **total** : un budget nul produit immediatement une action, aucun resultat ne depend d'une recherche de preuve non bornee.

Le pivot conceptuel general reste [`Dynamics-04-Open-Source-Game-Theory-Python.ipynb`](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) ; ce compagnon se concentre sur le modele borne et ne duplique pas son contenu.


**Versions jumelles :** [Python](Dynamics-05-Agents-Bornes-Python.ipynb) · [C#](Dynamics-05-Agents-Bornes-CSharp.ipynb).


## Ce que fait ce notebook - et ce qu'il ne fait pas

Aucune invocation Lean ne se produit ici : les noms et énoncés Lean sont des références. Le role du compagnon Rust est double :

1. **reproduire independamment** la famille finie de bots (`cooperateBot`, `defectBotBounded`, `mirrorBot`, `basicFamily`, `canonicalPD`) depuis zero, sans importer le code Lean ;
2. **rejouer les certificats calculables** du module et comparer chaque verdict Rust au theoreme Lean correspondant.

La distinction est fondamentale : le Rust vérifie une **matrice finie**. Dans le module cité, certains énoncés portent sur des cas fixés ; `defectBotBounded_unexploitable` quantifie les listes d’opposants, et `programNashCheck_eq_true` fournit séparément une équivalence générale. Un verdict Rust conforme ne remplace jamais la preuve formelle - il la rend **inspectable** : chaque ligne du tableau ci-dessous peut etre relue, modifiee, contredite par l'etudiant. Ce notebook ne formalise ni logique de prouvabilite ni theoreme de Lob, et n'extrapole pas vers Godel : le module Lean lui-meme s'en garde explicitement.


## Le code public et l'agent borne

Un `BoundedAgent` porte un code public inspectable (un des trois bots temoins) et un budget de raisonnement naturel. La traduction Rust est fidele a la structure Lean `ProgramCode` / `BoundedAgent` :


In [2]:
/// Code public d'un agent-programme (ProgramCode du module Lean).
#[derive(Clone, Copy, PartialEq, Eq)]
enum ProgramCode {
    CooperateBot,
    DefectBot,
    Mirror,
}

impl ProgramCode {
    /// Nom public du code, tel que le module Lean le denomme.
    fn name(self) -> &'static str {
        match self {
            ProgramCode::CooperateBot => "COOPERATE_BOT",
            ProgramCode::DefectBot => "DEFECT_BOT",
            ProgramCode::Mirror => "MIRROR",
        }
    }
}

/// Agent associant un code public a un budget de raisonnement fini.
#[derive(Clone, Copy, PartialEq, Eq)]
struct BoundedAgent {
    code: ProgramCode,
    budget: i64,
}

impl BoundedAgent {
    /// Constructeur : un code public et son budget de raisonnement.
    fn new(code: ProgramCode, budget: i64) -> Self {
        BoundedAgent { code, budget }
    }
}

/// Affichage `BoundedAgent(CODE, budget=n)`, comme le `__repr__` du jumeau Python.
impl std::fmt::Display for BoundedAgent {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        write!(f, "BoundedAgent({}, budget={})", self.code.name(), self.budget)
    }
}

// La famille temoin : un agent cooperateur, un agent defecteur, un agent miroir.
let cooperate_bot = BoundedAgent::new(ProgramCode::CooperateBot, 0);
let defect_bot_bounded = BoundedAgent::new(ProgramCode::DefectBot, 0);
let mirror_bot = BoundedAgent::new(ProgramCode::Mirror, 1);
let basic_family = vec![cooperate_bot, defect_bot_bounded, mirror_bot];

// La famille temoin, affichee dans l'ordre d'insertion (comme le jumeau Python).
println!("Famille temoin basicFamily :");
for agent in &basic_family {
    println!("  {}", agent);
}
// evcxr : la cellule se termine sur une instruction, il n'affiche rien apres.
let _ = ();


Famille temoin basicFamily :


  BoundedAgent(COOPERATE_BOT, budget=0)


  BoundedAgent(DEFECT_BOT, budget=0)


  BoundedAgent(MIRROR, budget=1)


**Lecture de la sortie — la famille temoin et ses budgets.** Trois lignes suffisent a poser le decor : `BoundedAgent(COOPERATE_BOT, budget=0)`, `BoundedAgent(DEFECT_BOT, budget=0)`, `BoundedAgent(MIRROR, budget=1)`. Les deux bots vivent a budget nul — leur decision ne regarde ni l'historique ni l'adversaire, aucune memoire a payer. Le miroir seul porte `budget=1` : inspecter le code adverse avec un budget positif est le test représenté, et c'est ce cout qui distinguera le miroir riche (b=1, il reflete) du miroir casse (b=0, l'exercice 1 y viendra) — la meme ligne de code, deux comportements selon un seul entier. Toute la question du notebook est deja la : que reste-t-il du jeu quand la lecture a un prix ?


## L'interprete total `act`

L'interprete ne fait aucune recherche de preuve : il inspecte le code adverse et le budget, puis produit une action. A **budget nul**, `mirror` deve (la règle à budget nul prescrit directement la défection) ; a **budget positif**, il coopere sauf contre le code explicitement defecteur :


In [3]:
#[derive(Clone, Copy, PartialEq, Eq)]
enum ProgramCode {
    CooperateBot,
    DefectBot,
    Mirror,
}

impl ProgramCode {
    fn name(self) -> &'static str {
        match self {
            ProgramCode::CooperateBot => "COOPERATE_BOT",
            ProgramCode::DefectBot => "DEFECT_BOT",
            ProgramCode::Mirror => "MIRROR",
        }
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
struct BoundedAgent {
    code: ProgramCode,
    budget: i64,
}

impl BoundedAgent {
    fn new(code: ProgramCode, budget: i64) -> Self {
        BoundedAgent { code, budget }
    }
}

impl std::fmt::Display for BoundedAgent {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        write!(f, "BoundedAgent({}, budget={})", self.code.name(), self.budget)
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
enum Action {
    Cooperate,
    Defect,
}

impl Action {
    fn name(self) -> &'static str {
        match self {
            Action::Cooperate => "cooperate",
            Action::Defect => "defect",
        }
    }

    fn initial(self) -> char {
        match self {
            Action::Cooperate => 'C',
            Action::Defect => 'D',
        }
    }
}

/// L'interpretateur total des codes publics (act du module Lean).
#[allow(dead_code)]
fn act(agent: &BoundedAgent, opponent: ProgramCode) -> Action {
    match agent.code {
        ProgramCode::CooperateBot => Action::Cooperate,
        ProgramCode::DefectBot => Action::Defect,
        ProgramCode::Mirror => {
            if agent.budget == 0 {
                Action::Defect
            } else if opponent == ProgramCode::DefectBot {
                Action::Defect
            } else {
                Action::Cooperate
            }
        }
    }
}

/// Resultat ordonne de l'interpretateur : (action de row, action de column).
#[allow(dead_code)]
fn outcome_bounded(row: &BoundedAgent, column: &BoundedAgent) -> (Action, Action) {
    (act(row, column.code), act(column, row.code))
}

/// Le tuple Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_tuple(a: Action, b: Action) -> String {
    format!("('{}', '{}')", a.name(), b.name())
}

/// Le booleen Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_bool(v: bool) -> String {
    if v { "True".to_string() } else { "False".to_string() }
}

let cooperate_bot = BoundedAgent::new(ProgramCode::CooperateBot, 0);
let defect_bot_bounded = BoundedAgent::new(ProgramCode::DefectBot, 0);
let mirror_bot = BoundedAgent::new(ProgramCode::Mirror, 1);
let basic_family = vec![cooperate_bot, defect_bot_bounded, mirror_bot];

for left in &basic_family {
    for right in &basic_family {
        let o = outcome_bounded(left, right);
        println!(
            "outcome_bounded({:<13} b={}, {:<13} b={}) = {}",
            left.code.name(),
            left.budget,
            right.code.name(),
            right.budget,
            py_tuple(o.0, o.1)
        );
    }
};


outcome_bounded(COOPERATE_BOT b=0, COOPERATE_BOT b=0) = ('cooperate', 'cooperate')


outcome_bounded(COOPERATE_BOT b=0, DEFECT_BOT    b=0) = ('cooperate', 'defect')


outcome_bounded(COOPERATE_BOT b=0, MIRROR        b=1) = ('cooperate', 'cooperate')


outcome_bounded(DEFECT_BOT    b=0, COOPERATE_BOT b=0) = ('defect', 'cooperate')


outcome_bounded(DEFECT_BOT    b=0, DEFECT_BOT    b=0) = ('defect', 'defect')


outcome_bounded(DEFECT_BOT    b=0, MIRROR        b=1) = ('defect', 'defect')


outcome_bounded(MIRROR        b=1, COOPERATE_BOT b=0) = ('cooperate', 'cooperate')


outcome_bounded(MIRROR        b=1, DEFECT_BOT    b=0) = ('defect', 'defect')


outcome_bounded(MIRROR        b=1, MIRROR        b=1) = ('cooperate', 'cooperate')


**Lecture chiffree — les issues bornees, ligne par ligne.** La sortie egraine `outcome_bounded` sur chaque paire. Les lignes COOPERATE_BOT : `(cooperate, cooperate)` contre son semblable, `(cooperate, defect)` contre DEFECT_BOT — le coopereur ne s'adapte pas, il subit. Les lignes DEFECT_BOT font le symetrique : `(defect, cooperate)` puis `(defect, defect)`. La ligne qui instruit est la troisieme colonne : contre `MIRROR b=1`, COOPERATE_BOT obtient `('cooperate', 'cooperate')` — le miroir inspecte une fois le code du coopérateur et coopère ; DEFECT_BOT contre le miroir obtient la paire symetrique de defection : le miroir détecte le code défecteur et dévie. Deux bots a budget nul ont des issues independantes du budget ; le miroir a b=1 est le seul dont l'issue depende de ce qu'il a pu lire — la famille temoin deliberate ce contraste.


La matrice des issues de la famille temoin se lit d'un coup d'oeil :


In [4]:
#[derive(Clone, Copy, PartialEq, Eq)]
enum ProgramCode {
    CooperateBot,
    DefectBot,
    Mirror,
}

impl ProgramCode {
    fn name(self) -> &'static str {
        match self {
            ProgramCode::CooperateBot => "COOPERATE_BOT",
            ProgramCode::DefectBot => "DEFECT_BOT",
            ProgramCode::Mirror => "MIRROR",
        }
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
struct BoundedAgent {
    code: ProgramCode,
    budget: i64,
}

impl BoundedAgent {
    fn new(code: ProgramCode, budget: i64) -> Self {
        BoundedAgent { code, budget }
    }
}

impl std::fmt::Display for BoundedAgent {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        write!(f, "BoundedAgent({}, budget={})", self.code.name(), self.budget)
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
enum Action {
    Cooperate,
    Defect,
}

impl Action {
    fn name(self) -> &'static str {
        match self {
            Action::Cooperate => "cooperate",
            Action::Defect => "defect",
        }
    }

    fn initial(self) -> char {
        match self {
            Action::Cooperate => 'C',
            Action::Defect => 'D',
        }
    }
}

/// L'interpretateur total des codes publics (act du module Lean).
#[allow(dead_code)]
fn act(agent: &BoundedAgent, opponent: ProgramCode) -> Action {
    match agent.code {
        ProgramCode::CooperateBot => Action::Cooperate,
        ProgramCode::DefectBot => Action::Defect,
        ProgramCode::Mirror => {
            if agent.budget == 0 {
                Action::Defect
            } else if opponent == ProgramCode::DefectBot {
                Action::Defect
            } else {
                Action::Cooperate
            }
        }
    }
}

/// Resultat ordonne de l'interpretateur : (action de row, action de column).
#[allow(dead_code)]
fn outcome_bounded(row: &BoundedAgent, column: &BoundedAgent) -> (Action, Action) {
    (act(row, column.code), act(column, row.code))
}

let cooperate_bot = BoundedAgent::new(ProgramCode::CooperateBot, 0);
let defect_bot_bounded = BoundedAgent::new(ProgramCode::DefectBot, 0);
let mirror_bot = BoundedAgent::new(ProgramCode::Mirror, 1);
let basic_family = vec![cooperate_bot, defect_bot_bounded, mirror_bot];

/// Le nom tronque a neuf caracteres, comme `name[:9]` en Python.
#[allow(dead_code)]
fn name9(name: &str) -> &str {
    &name[..name.len().min(9)]
}

println!("Matrice des issues de basicFamily (ligne = agent, colonne = adversaire)\n");
let header = basic_family
    .iter()
    .map(|a| format!("{:>12}", name9(a.code.name())))
    .collect::<String>();
println!("{:>14}{}", "", header);
for row in &basic_family {
    let mut cells_row = String::new();
    for col in &basic_family {
        let o = outcome_bounded(row, col);
        cells_row.push_str(&format!("{}/{:>4}", o.0.initial(), o.1.initial()));
    }
    println!("{:>12} b={} {}", name9(row.code.name()), row.budget, cells_row);
}
println!("\nC = cooperate, D = defect (format : action de l'agent en ligne / action de l'adversaire)");


Matrice des issues de basicFamily (ligne = agent, colonne = adversaire)


                 COOPERATE   DEFECT_BO      MIRROR


   COOPERATE b=0 C/   CC/   DC/   C


   DEFECT_BO b=0 D/   CD/   DD/   D


      MIRROR b=1 C/   CD/   DC/   C


C = cooperate, D = defect (format : action de l'agent en ligne / action de l'adversaire)


**Lecture de la matrice — trois lignes, trois caracteres.** Le tableau croise condense la sortie precedente : ligne = agent, colonne = adversaire, format `action de l'agent / action de l'adversaire`. La ligne COOPERATE_BOT affiche trois `C/` : il coopere contre tous, sans exception ni memoire. La ligne DEFECT_BOT affiche trois `D/` : defection systématique, le complement exact. La ligne MIRROR dépend du code public de son adversaire : `C/C` face au coopereur, `D/D` face au defecteur, `C/C` face a lui-meme — ce résultat provient du test sur les trois codes, sans historique ni exécution itérée. Cette coïncidence des issues est limitée à cette famille finie.


## Le dilemme du prisonnier canonique et le rang de paiement

Le parametrage canonique est `T=5, R=3, P=1, S=0`. Le module definit un **rang de paiement fini** `payoffRank` (`CC=3, CD=0, DC=5, DD=1`) et prouve (`payoffRank_le_iff`) que ce rang preserve exactement l'ordre des paiements du jeu canonique - evitant de comparer des reels arbitraires :


In [5]:
#[derive(Clone, Copy, PartialEq, Eq)]
enum ProgramCode {
    CooperateBot,
    DefectBot,
    Mirror,
}

impl ProgramCode {
    fn name(self) -> &'static str {
        match self {
            ProgramCode::CooperateBot => "COOPERATE_BOT",
            ProgramCode::DefectBot => "DEFECT_BOT",
            ProgramCode::Mirror => "MIRROR",
        }
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
struct BoundedAgent {
    code: ProgramCode,
    budget: i64,
}

impl BoundedAgent {
    fn new(code: ProgramCode, budget: i64) -> Self {
        BoundedAgent { code, budget }
    }
}

impl std::fmt::Display for BoundedAgent {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        write!(f, "BoundedAgent({}, budget={})", self.code.name(), self.budget)
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
enum Action {
    Cooperate,
    Defect,
}

impl Action {
    fn name(self) -> &'static str {
        match self {
            Action::Cooperate => "cooperate",
            Action::Defect => "defect",
        }
    }

    fn initial(self) -> char {
        match self {
            Action::Cooperate => 'C',
            Action::Defect => 'D',
        }
    }
}

/// Le tuple Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_tuple(a: Action, b: Action) -> String {
    format!("('{}', '{}')", a.name(), b.name())
}

/// Le booleen Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_bool(v: bool) -> String {
    if v { "True".to_string() } else { "False".to_string() }
}

const PD_KEYS: [&str; 4] = ["T", "R", "P", "S"];
const PD_VALS: [i64; 4] = [5, 3, 1, 0];

#[allow(dead_code)]
fn pd(key: &str) -> i64 {
    for i in 0..PD_KEYS.len() {
        if PD_KEYS[i] == key {
            return PD_VALS[i];
        }
    }
    0
}

/// Paiement de l'agent ligne dans le PD canonique.
#[allow(dead_code)]
fn stage_payoff(row_action: Action, col_action: Action) -> i64 {
    if row_action == Action::Cooperate && col_action == Action::Cooperate {
        pd("R")
    } else if row_action == Action::Cooperate {
        pd("S")
    } else if col_action == Action::Cooperate {
        pd("T")
    } else {
        pd("P")
    }
}

/// Rang fini du module Lean : CC=3, CD=0, DC=5, DD=1.
#[allow(dead_code)]
fn payoff_rank(row_action: Action, col_action: Action) -> i64 {
    match (row_action, col_action) {
        (Action::Cooperate, Action::Cooperate) => 3,
        (Action::Cooperate, Action::Defect) => 0,
        (Action::Defect, Action::Cooperate) => 5,
        (Action::Defect, Action::Defect) => 1,
    }
}

let dict = PD_KEYS
    .iter()
    .zip(PD_VALS.iter())
    .map(|(k, v)| format!("'{}': {}", k, v))
    .collect::<Vec<_>>()
    .join(", ");
println!("PD canonique : {{{}}}", dict);
for pair in [
    (Action::Cooperate, Action::Cooperate),
    (Action::Cooperate, Action::Defect),
    (Action::Defect, Action::Cooperate),
    (Action::Defect, Action::Defect),
] {
    println!(
        "  {} -> stage_payoff={}, payoffRank={}",
        py_tuple(pair.0, pair.1),
        stage_payoff(pair.0, pair.1),
        payoff_rank(pair.0, pair.1)
    );
};


PD canonique : {'T': 5, 'R': 3, 'P': 1, 'S': 0}


  ('cooperate', 'cooperate') -> stage_payoff=3, payoffRank=3


  ('cooperate', 'defect') -> stage_payoff=0, payoffRank=0


  ('defect', 'cooperate') -> stage_payoff=5, payoffRank=5


  ('defect', 'defect') -> stage_payoff=1, payoffRank=1


**Lecture chiffree — le rang de paiement encode l'ordre du dilemme.** La sortie applique le PD canonique `{'T': 5, 'R': 3, 'P': 1, 'S': 0}` aux quatre issues. Chaque ligne porte deux nombres : `('cooperate','cooperate') -> stage_payoff=3, payoffRank=3`, `('cooperate','defect') -> 0, 0`, `('defect','cooperate') -> 5, 5`, `('defect','defect') -> 1, 1`. Ici rang et gain coïncident chiffre à chiffre parce que `payoffRank` reprend explicitement les quatre valeurs canoniques, pas simplement parce qu'elles sont distinctes. Il préserve l'ordre T > R > P > S pour ce jeu ; cette vérification n'établit aucune comparabilité cardinale entre jeux différents. Le certificat final du notebook (16 paires) portera precisement sur ce rang.


## Les organes calculables

Le module expose trois organes booléens — `mutualCooperationCheck`, `unexploitableCheck` et `programNashCheck` — correspondant aux définitions propositionnelles. L’équivalence générale `programNashCheck_eq_true` est explicitement prouvée dans Lean pour le PD canonique. Le calcul ci-dessous reproduit ces organes sur les familles finies du carnet ; aucune invocation Lean n’a lieu ici.


In [6]:
#[derive(Clone, Copy, PartialEq, Eq)]
enum ProgramCode {
    CooperateBot,
    DefectBot,
    Mirror,
}

impl ProgramCode {
    fn name(self) -> &'static str {
        match self {
            ProgramCode::CooperateBot => "COOPERATE_BOT",
            ProgramCode::DefectBot => "DEFECT_BOT",
            ProgramCode::Mirror => "MIRROR",
        }
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
struct BoundedAgent {
    code: ProgramCode,
    budget: i64,
}

impl BoundedAgent {
    fn new(code: ProgramCode, budget: i64) -> Self {
        BoundedAgent { code, budget }
    }
}

impl std::fmt::Display for BoundedAgent {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        write!(f, "BoundedAgent({}, budget={})", self.code.name(), self.budget)
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
enum Action {
    Cooperate,
    Defect,
}

impl Action {
    fn name(self) -> &'static str {
        match self {
            Action::Cooperate => "cooperate",
            Action::Defect => "defect",
        }
    }

    fn initial(self) -> char {
        match self {
            Action::Cooperate => 'C',
            Action::Defect => 'D',
        }
    }
}

/// L'interpretateur total des codes publics (act du module Lean).
#[allow(dead_code)]
fn act(agent: &BoundedAgent, opponent: ProgramCode) -> Action {
    match agent.code {
        ProgramCode::CooperateBot => Action::Cooperate,
        ProgramCode::DefectBot => Action::Defect,
        ProgramCode::Mirror => {
            if agent.budget == 0 {
                Action::Defect
            } else if opponent == ProgramCode::DefectBot {
                Action::Defect
            } else {
                Action::Cooperate
            }
        }
    }
}

/// Resultat ordonne de l'interpretateur : (action de row, action de column).
#[allow(dead_code)]
fn outcome_bounded(row: &BoundedAgent, column: &BoundedAgent) -> (Action, Action) {
    (act(row, column.code), act(column, row.code))
}

let cooperate_bot = BoundedAgent::new(ProgramCode::CooperateBot, 0);
let defect_bot_bounded = BoundedAgent::new(ProgramCode::DefectBot, 0);
let mirror_bot = BoundedAgent::new(ProgramCode::Mirror, 1);
let basic_family = vec![cooperate_bot, defect_bot_bounded, mirror_bot];

const PD_KEYS: [&str; 4] = ["T", "R", "P", "S"];
const PD_VALS: [i64; 4] = [5, 3, 1, 0];

#[allow(dead_code)]
fn pd(key: &str) -> i64 {
    for i in 0..PD_KEYS.len() {
        if PD_KEYS[i] == key {
            return PD_VALS[i];
        }
    }
    0
}

/// Paiement de l'agent ligne dans le PD canonique.
#[allow(dead_code)]
fn stage_payoff(row_action: Action, col_action: Action) -> i64 {
    if row_action == Action::Cooperate && col_action == Action::Cooperate {
        pd("R")
    } else if row_action == Action::Cooperate {
        pd("S")
    } else if col_action == Action::Cooperate {
        pd("T")
    } else {
        pd("P")
    }
}

/// Rang fini du module Lean : CC=3, CD=0, DC=5, DD=1.
#[allow(dead_code)]
fn payoff_rank(row_action: Action, col_action: Action) -> i64 {
    match (row_action, col_action) {
        (Action::Cooperate, Action::Cooperate) => 3,
        (Action::Cooperate, Action::Defect) => 0,
        (Action::Defect, Action::Cooperate) => 5,
        (Action::Defect, Action::Defect) => 1,
    }
}

/// Organe de cooperation mutuelle (mutualCooperationCheck).
#[allow(dead_code)]
fn mutual_cooperation_check(left: &BoundedAgent, right: &BoundedAgent) -> bool {
    outcome_bounded(left, right) == (Action::Cooperate, Action::Cooperate)
}

/// Organe d'inexploitabilite sur une famille finie (unexploitableCheck).
#[allow(dead_code)]
fn unexploitable_check(agent: &BoundedAgent, opponents: &[BoundedAgent]) -> bool {
    opponents
        .iter()
        .all(|opp| outcome_bounded(agent, opp) != (Action::Cooperate, Action::Defect))
}

/// Organe d'equilibre relatif pour le PD canonique (programNashCheck).
#[allow(dead_code)]
fn program_nash_check(
    family: &[BoundedAgent],
    left: &BoundedAgent,
    right: &BoundedAgent,
) -> bool {
    let base = outcome_bounded(left, right);
    let base_left = payoff_rank(base.0, base.1);
    let base_right = payoff_rank(base.1, base.0);
    let left_ok = family.iter().all(|alt| {
        let o = outcome_bounded(alt, right);
        payoff_rank(o.0, o.1) <= base_left
    });
    let right_ok = family.iter().all(|alt| {
        let o = outcome_bounded(left, alt);
        payoff_rank(o.1, o.0) <= base_right
    });
    left_ok && right_ok
}

// Aucun appel : cette cellule pose les trois organes calculables.


## Exercice 1 - Le miroir sans budget

Le module documente qu'a budget nul, `mirror` deve. On construit `mirror_broke = BoundedAgent(ProgramCode.MIRROR, 0)`. **Question** : ce miroir sans budget est-il inexploitable dans `basic_family` ? La cooperation mutuelle est-elle encore possible ? Completer la fonction pour calculer les deux verdicts, puis verifier contre l'intuition : un miroir a budget nul se comporte structurellement comme quel bot de la famille ?


In [7]:
#[derive(Clone, Copy, PartialEq, Eq)]
enum ProgramCode {
    CooperateBot,
    DefectBot,
    Mirror,
}

impl ProgramCode {
    fn name(self) -> &'static str {
        match self {
            ProgramCode::CooperateBot => "COOPERATE_BOT",
            ProgramCode::DefectBot => "DEFECT_BOT",
            ProgramCode::Mirror => "MIRROR",
        }
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
struct BoundedAgent {
    code: ProgramCode,
    budget: i64,
}

impl BoundedAgent {
    fn new(code: ProgramCode, budget: i64) -> Self {
        BoundedAgent { code, budget }
    }
}

impl std::fmt::Display for BoundedAgent {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        write!(f, "BoundedAgent({}, budget={})", self.code.name(), self.budget)
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
enum Action {
    Cooperate,
    Defect,
}

impl Action {
    fn name(self) -> &'static str {
        match self {
            Action::Cooperate => "cooperate",
            Action::Defect => "defect",
        }
    }

    fn initial(self) -> char {
        match self {
            Action::Cooperate => 'C',
            Action::Defect => 'D',
        }
    }
}

/// L'interpretateur total des codes publics (act du module Lean).
#[allow(dead_code)]
fn act(agent: &BoundedAgent, opponent: ProgramCode) -> Action {
    match agent.code {
        ProgramCode::CooperateBot => Action::Cooperate,
        ProgramCode::DefectBot => Action::Defect,
        ProgramCode::Mirror => {
            if agent.budget == 0 {
                Action::Defect
            } else if opponent == ProgramCode::DefectBot {
                Action::Defect
            } else {
                Action::Cooperate
            }
        }
    }
}

/// Resultat ordonne de l'interpretateur : (action de row, action de column).
#[allow(dead_code)]
fn outcome_bounded(row: &BoundedAgent, column: &BoundedAgent) -> (Action, Action) {
    (act(row, column.code), act(column, row.code))
}

let cooperate_bot = BoundedAgent::new(ProgramCode::CooperateBot, 0);
let defect_bot_bounded = BoundedAgent::new(ProgramCode::DefectBot, 0);
let mirror_bot = BoundedAgent::new(ProgramCode::Mirror, 1);
let basic_family = vec![cooperate_bot, defect_bot_bounded, mirror_bot];

/// Organe de cooperation mutuelle (mutualCooperationCheck).
#[allow(dead_code)]
fn mutual_cooperation_check(left: &BoundedAgent, right: &BoundedAgent) -> bool {
    outcome_bounded(left, right) == (Action::Cooperate, Action::Cooperate)
}

/// Organe d'inexploitabilite sur une famille finie (unexploitableCheck).
#[allow(dead_code)]
fn unexploitable_check(agent: &BoundedAgent, opponents: &[BoundedAgent]) -> bool {
    opponents
        .iter()
        .all(|opp| outcome_bounded(agent, opp) != (Action::Cooperate, Action::Defect))
}

/// Organe d'equilibre relatif pour le PD canonique (programNashCheck).
#[allow(dead_code)]
fn program_nash_check(
    family: &[BoundedAgent],
    left: &BoundedAgent,
    right: &BoundedAgent,
) -> bool {
    let base = outcome_bounded(left, right);
    let base_left = payoff_rank(base.0, base.1);
    let base_right = payoff_rank(base.1, base.0);
    let left_ok = family.iter().all(|alt| {
        let o = outcome_bounded(alt, right);
        payoff_rank(o.0, o.1) <= base_left
    });
    let right_ok = family.iter().all(|alt| {
        let o = outcome_bounded(left, alt);
        payoff_rank(o.1, o.0) <= base_right
    });
    left_ok && right_ok
}

let mirror_broke = BoundedAgent::new(ProgramCode::Mirror, 0);

/// Renvoie (inexploitable, cooperation_mutuelle_avec_lui_meme).
/// TODO Etudiant : utiliser unexploitable_check et mutual_cooperation_check
/// sur mirror_broke (contre basic_family puis contre lui-meme).
#[allow(dead_code)]
fn verdicts_miroir_sans_budget(
    _agent: &BoundedAgent,
    _famille: &[BoundedAgent],
) -> Option<(bool, bool)> {
    None // TODO etudiant
}

println!("Exercice 1 a completer : verdicts_miroir_sans_budget()");
println!("Indice : budget nul => act(MIRROR, _) renvoie toujours...");

// Les copies isolent la famille ; le garde verifie aussi des miroirs a budget positif.
let _cas1: Vec<(BoundedAgent, Vec<BoundedAgent>)> = vec![
    (mirror_broke, basic_family.clone()),
    (mirror_bot, basic_family.clone()),
    (cooperate_bot, vec![defect_bot_bounded]),
];
let _attendus1: Vec<(bool, bool)> = _cas1
    .iter()
    .map(|(a, f)| (unexploitable_check(a, f), mutual_cooperation_check(a, a)))
    .collect();
let _reps1: Vec<Option<(bool, bool)>> = _cas1
    .iter()
    .map(|(a, f)| verdicts_miroir_sans_budget(a, f))
    .collect();
assert!(
    _reps1.iter().all(|r| r.is_none()) || _reps1.iter().all(|r| r.is_some()),
    "Reponse partielle"
);
if _reps1[0].is_some() {
    assert!(
        _reps1
            .iter()
            .zip(_attendus1.iter())
            .all(|(r, e)| r.as_ref() == Some(e)),
        "Verdicts incorrects"
    );
};


Exercice 1 a completer : verdicts_miroir_sans_budget()


Indice : budget nul => act(MIRROR, _) renvoie toujours...


## Exercice 2 - Le paiement du miroir contre le defecteur

Le theoreme `mirror_basicFamily_unexploitable` affirme que le miroir a budget positif est inexploitable dans la famille temoin : il deve contre `defectBotBounded`, cooperant avec les autres. **Question** : quel paiement le miroir obtient-il effectivement contre chaque membre de la famille, et quel membre est son pire adversaire ? Completer le calcul du dict des paiements et du pire adversaire, puis conclure : inexploitable signifie-t-il optimal ?


In [8]:
#[derive(Clone, Copy, PartialEq, Eq)]
enum ProgramCode {
    CooperateBot,
    DefectBot,
    Mirror,
}

impl ProgramCode {
    fn name(self) -> &'static str {
        match self {
            ProgramCode::CooperateBot => "COOPERATE_BOT",
            ProgramCode::DefectBot => "DEFECT_BOT",
            ProgramCode::Mirror => "MIRROR",
        }
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
struct BoundedAgent {
    code: ProgramCode,
    budget: i64,
}

impl BoundedAgent {
    fn new(code: ProgramCode, budget: i64) -> Self {
        BoundedAgent { code, budget }
    }
}

impl std::fmt::Display for BoundedAgent {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        write!(f, "BoundedAgent({}, budget={})", self.code.name(), self.budget)
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
enum Action {
    Cooperate,
    Defect,
}

impl Action {
    fn name(self) -> &'static str {
        match self {
            Action::Cooperate => "cooperate",
            Action::Defect => "defect",
        }
    }

    fn initial(self) -> char {
        match self {
            Action::Cooperate => 'C',
            Action::Defect => 'D',
        }
    }
}

/// L'interpretateur total des codes publics (act du module Lean).
#[allow(dead_code)]
fn act(agent: &BoundedAgent, opponent: ProgramCode) -> Action {
    match agent.code {
        ProgramCode::CooperateBot => Action::Cooperate,
        ProgramCode::DefectBot => Action::Defect,
        ProgramCode::Mirror => {
            if agent.budget == 0 {
                Action::Defect
            } else if opponent == ProgramCode::DefectBot {
                Action::Defect
            } else {
                Action::Cooperate
            }
        }
    }
}

/// Resultat ordonne de l'interpretateur : (action de row, action de column).
#[allow(dead_code)]
fn outcome_bounded(row: &BoundedAgent, column: &BoundedAgent) -> (Action, Action) {
    (act(row, column.code), act(column, row.code))
}

/// Le tuple Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_tuple(a: Action, b: Action) -> String {
    format!("('{}', '{}')", a.name(), b.name())
}

/// Le booleen Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_bool(v: bool) -> String {
    if v { "True".to_string() } else { "False".to_string() }
}

let cooperate_bot = BoundedAgent::new(ProgramCode::CooperateBot, 0);
let defect_bot_bounded = BoundedAgent::new(ProgramCode::DefectBot, 0);
let mirror_bot = BoundedAgent::new(ProgramCode::Mirror, 1);
let basic_family = vec![cooperate_bot, defect_bot_bounded, mirror_bot];

const PD_KEYS: [&str; 4] = ["T", "R", "P", "S"];
const PD_VALS: [i64; 4] = [5, 3, 1, 0];

#[allow(dead_code)]
fn pd(key: &str) -> i64 {
    for i in 0..PD_KEYS.len() {
        if PD_KEYS[i] == key {
            return PD_VALS[i];
        }
    }
    0
}

/// Paiement de l'agent ligne dans le PD canonique.
#[allow(dead_code)]
fn stage_payoff(row_action: Action, col_action: Action) -> i64 {
    if row_action == Action::Cooperate && col_action == Action::Cooperate {
        pd("R")
    } else if row_action == Action::Cooperate {
        pd("S")
    } else if col_action == Action::Cooperate {
        pd("T")
    } else {
        pd("P")
    }
}

/// Rang fini du module Lean : CC=3, CD=0, DC=5, DD=1.
#[allow(dead_code)]
fn payoff_rank(row_action: Action, col_action: Action) -> i64 {
    match (row_action, col_action) {
        (Action::Cooperate, Action::Cooperate) => 3,
        (Action::Cooperate, Action::Defect) => 0,
        (Action::Defect, Action::Cooperate) => 5,
        (Action::Defect, Action::Defect) => 1,
    }
}

/// Renvoie (paiements {adversaire: paiement}, pire_adversaire).
/// TODO Etudiant : pour chaque agent de basic_family, calculer
/// stage_payoff(*outcome_bounded(mirror_bot, adversaire))
/// (attention : outcome_bounded renvoie (action miroir, action adversaire)).
/// Le dictionnaire Python est indexe par agent ; ici l'index dans la famille
/// tient lieu de cle (Rust n'a pas d'ordre d'iteration arbitraire a preserver).
#[allow(dead_code)]
fn paiements_miroir(
    _agent: &BoundedAgent,
    _famille: &[BoundedAgent],
) -> Option<(Vec<(usize, i64)>, usize)> {
    None // TODO etudiant
}

println!("Exercice 2 a completer : paiements_miroir()");
println!("Indice : inexploitable protege du pire (S=0), pas de l'optimum (T=5).");

// L'oracle conserve les references intactes et verifie paiements et minimum.
let _cas2: Vec<(BoundedAgent, Vec<BoundedAgent>)> = vec![
    (mirror_bot, basic_family.clone()),
    (mirror_broke, basic_family.clone()),
    (cooperate_bot, vec![defect_bot_bounded]),
];
let _attendus2: Vec<(Vec<(usize, i64)>, usize)> = _cas2
    .iter()
    .map(|(_agent, _famille)| {
        let _gains: Vec<(usize, i64)> = _famille
            .iter()
            .enumerate()
            .map(|(i, opp)| {
                let o = outcome_bounded(_agent, opp);
                (i, stage_payoff(o.0, o.1))
            })
            .collect();
        let _pire = _gains.iter().enumerate().fold(0usize, |acc, (i, (_, g))| {
            if *g < _gains[acc].1 {
                i
            } else {
                acc
            }
        });
        (_gains, _pire)
    })
    .collect();
let _reps2: Vec<Option<(Vec<(usize, i64)>, usize)>> = _cas2
    .iter()
    .map(|(a, f)| paiements_miroir(a, f))
    .collect();
assert!(
    _reps2.iter().all(|r| r.is_none()) || _reps2.iter().all(|r| r.is_some()),
    "Reponse partielle"
);
if _reps2[0].is_some() {
    assert!(
        _reps2
            .iter()
            .zip(_attendus2.iter())
            .all(|(r, e)| r.as_ref() == Some(e)),
        "Paiement ou pire adversaire incorrect"
    );
};


Exercice 2 a completer : paiements_miroir()


Indice : inexploitable protege du pire (S=0), pas de l'optimum (T=5).


## Rejeu des certificats Lean

Les huit lignes ci-dessous confrontent des calculs finis à des énoncés Lean cités : plusieurs sont des cas fixés de la famille témoin, et la colonne rappelle leur référence. `defectBotBounded_unexploitable` quantifie les listes d’opposants ; `payoffRank_le_iff` compare les 16 couples d’issues du seul `canonicalPD`. Les lignes 6 et 7 calculent le même profil, sous forme propositionnelle puis booléenne. L’équivalence générale supplémentaire `programNashCheck_eq_true` est distincte de ces huit résultats. Aucun Lean n’est exécuté ici.


**Protocole de lecture — decoder une ligne de certificat.** Chaque ligne de sortie a suivre a la meme grammaire : un nom court (`mirror_mirror`), un verdict `Python=True`, une fleche `<-`, puis l'enonce Lean dont c'est la replique (le libelle `Python=` est celui du tableau de reference : dans ce carnet Rust, c'est le calcul Rust qui etablit le verdict) (`MutualCooperationBounded mirrorBot mirrorBot`). Trois choses a ne pas confondre : le verdict `True` dit que LE CALCUL de ce carnet (Rust) rend vrai sur la famille temoin, pas que l'énoncé Lean cité a été vérifié par ce carnet ; le nom court est une cle de lecture, pas l'enonce ; et la fleche pointe de la preuve calculee vers la cible formelle — le sens du rejeu. Les enonces quantifies (sur toute une famille d'adversaires) se lisent differemment des egalites simples : l'interpretation qui suit cette section detaille ce partage.


In [9]:
#[derive(Clone, Copy, PartialEq, Eq)]
enum ProgramCode {
    CooperateBot,
    DefectBot,
    Mirror,
}

impl ProgramCode {
    fn name(self) -> &'static str {
        match self {
            ProgramCode::CooperateBot => "COOPERATE_BOT",
            ProgramCode::DefectBot => "DEFECT_BOT",
            ProgramCode::Mirror => "MIRROR",
        }
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
struct BoundedAgent {
    code: ProgramCode,
    budget: i64,
}

impl BoundedAgent {
    fn new(code: ProgramCode, budget: i64) -> Self {
        BoundedAgent { code, budget }
    }
}

impl std::fmt::Display for BoundedAgent {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        write!(f, "BoundedAgent({}, budget={})", self.code.name(), self.budget)
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
enum Action {
    Cooperate,
    Defect,
}

impl Action {
    fn name(self) -> &'static str {
        match self {
            Action::Cooperate => "cooperate",
            Action::Defect => "defect",
        }
    }

    fn initial(self) -> char {
        match self {
            Action::Cooperate => 'C',
            Action::Defect => 'D',
        }
    }
}

/// L'interpretateur total des codes publics (act du module Lean).
#[allow(dead_code)]
fn act(agent: &BoundedAgent, opponent: ProgramCode) -> Action {
    match agent.code {
        ProgramCode::CooperateBot => Action::Cooperate,
        ProgramCode::DefectBot => Action::Defect,
        ProgramCode::Mirror => {
            if agent.budget == 0 {
                Action::Defect
            } else if opponent == ProgramCode::DefectBot {
                Action::Defect
            } else {
                Action::Cooperate
            }
        }
    }
}

/// Resultat ordonne de l'interpretateur : (action de row, action de column).
#[allow(dead_code)]
fn outcome_bounded(row: &BoundedAgent, column: &BoundedAgent) -> (Action, Action) {
    (act(row, column.code), act(column, row.code))
}

/// Le tuple Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_tuple(a: Action, b: Action) -> String {
    format!("('{}', '{}')", a.name(), b.name())
}

/// Le booleen Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_bool(v: bool) -> String {
    if v { "True".to_string() } else { "False".to_string() }
}

let cooperate_bot = BoundedAgent::new(ProgramCode::CooperateBot, 0);
let defect_bot_bounded = BoundedAgent::new(ProgramCode::DefectBot, 0);
let mirror_bot = BoundedAgent::new(ProgramCode::Mirror, 1);
let basic_family = vec![cooperate_bot, defect_bot_bounded, mirror_bot];

/// Organe de cooperation mutuelle (mutualCooperationCheck).
#[allow(dead_code)]
fn mutual_cooperation_check(left: &BoundedAgent, right: &BoundedAgent) -> bool {
    outcome_bounded(left, right) == (Action::Cooperate, Action::Cooperate)
}

/// Organe d'inexploitabilite sur une famille finie (unexploitableCheck).
#[allow(dead_code)]
fn unexploitable_check(agent: &BoundedAgent, opponents: &[BoundedAgent]) -> bool {
    opponents
        .iter()
        .all(|opp| outcome_bounded(agent, opp) != (Action::Cooperate, Action::Defect))
}

/// Organe d'equilibre relatif pour le PD canonique (programNashCheck).
#[allow(dead_code)]
fn program_nash_check(
    family: &[BoundedAgent],
    left: &BoundedAgent,
    right: &BoundedAgent,
) -> bool {
    let base = outcome_bounded(left, right);
    let base_left = payoff_rank(base.0, base.1);
    let base_right = payoff_rank(base.1, base.0);
    let left_ok = family.iter().all(|alt| {
        let o = outcome_bounded(alt, right);
        payoff_rank(o.0, o.1) <= base_left
    });
    let right_ok = family.iter().all(|alt| {
        let o = outcome_bounded(left, alt);
        payoff_rank(o.1, o.0) <= base_right
    });
    left_ok && right_ok
}

#[allow(dead_code)]
fn enregistrer(
    nom: &str,
    verdict: bool,
    theoreme_lean: &str,
    certificats: &mut Vec<(String, bool, String)>,
) {
    certificats.push((nom.to_string(), verdict, theoreme_lean.to_string()));
}

#[allow(dead_code)]
fn verifier(
    nom: &str,
    verdict: bool,
    theoreme_lean: &str,
    certificats: &mut Vec<(String, bool, String)>,
) {
    certificats.push((nom.to_string(), verdict, theoreme_lean.to_string()));
    println!("{:<42} Python={:<5}  <-  {}", nom, py_bool(verdict), theoreme_lean);
}

let mut certificats: Vec<(String, bool, String)> = Vec::new();

// 1. Deux bots cooperateurs produisent la cooperation mutuelle
verifier(
    "cooperate_cooperate",
    mutual_cooperation_check(&cooperate_bot, &cooperate_bot),
    "MutualCooperationBounded cooperateBot cooperateBot",
    &mut certificats,
);

// 2. Deux bots defecteurs produisent la defection mutuelle
verifier(
    "defect_defect",
    outcome_bounded(&defect_bot_bounded, &defect_bot_bounded) == (Action::Defect, Action::Defect),
    "outcomeBounded defectBotBounded defectBotBounded = (defect, defect)",
    &mut certificats,
);


cooperate_cooperate                        Python=True   <-  MutualCooperationBounded cooperateBot cooperateBot


defect_defect                              Python=True   <-  outcomeBounded defectBotBounded defectBotBounded = (defect, defect)


**Lecture chiffree — les deux premieres egalites calculees.** `cooperate_cooperate Python=True <- MutualCooperationBounded cooperateBot cooperateBot` : les deux coopereurs bornes cooperent mutuellement (verdict calcule par ce carnet Rust) — une egalite que la matrice de la section precedente montrait deja case par case (`C/ C` en haut a gauche), ici confrontee a l'enonce exact. `defect_defect Python=True <- outcomeBounded defectBotBounded defectBotBounded = (defect, defect)` : la paire de defecteurs se defie mutuellement, meme egalite sur l'autre coin diagonal. Les deux certificats bornent la matrice par ses deux extremites pures — cooperation universelle d'un cote, defection universelle de l'autre — avant que les suivants n'attaquent les cas ou la strategie depend de l'autre.


### Ce que chaque certificat engage

Les deux premiers certificats sont des **egalites calculees** sur la matrice finie : on evalue une decision et on la compare au membre droit de l'enonce Lean. Les suivants melangent deux natures : des egalites du meme genre, et un enonce **quantifie** (`UnexploitableInFamily defectBotBounded opponents`), dont le verdict ne se lit plus cellule par cellule mais sur **toute** la famille d'adversaires.

Un calcul Rust peut *corroborer* un enonce quantifie sur la famille temoin ; il ne peut pas l'**etablir** pour une famille arbitraire. C'est le partage des roles que ce rejeu met en scene : le calcul exhibe un contre-exemple s'il en existe un, la preuve Lean fournit la quantification.

In [10]:
#[derive(Clone, Copy, PartialEq, Eq)]
enum ProgramCode {
    CooperateBot,
    DefectBot,
    Mirror,
}

impl ProgramCode {
    fn name(self) -> &'static str {
        match self {
            ProgramCode::CooperateBot => "COOPERATE_BOT",
            ProgramCode::DefectBot => "DEFECT_BOT",
            ProgramCode::Mirror => "MIRROR",
        }
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
struct BoundedAgent {
    code: ProgramCode,
    budget: i64,
}

impl BoundedAgent {
    fn new(code: ProgramCode, budget: i64) -> Self {
        BoundedAgent { code, budget }
    }
}

impl std::fmt::Display for BoundedAgent {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        write!(f, "BoundedAgent({}, budget={})", self.code.name(), self.budget)
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
enum Action {
    Cooperate,
    Defect,
}

impl Action {
    fn name(self) -> &'static str {
        match self {
            Action::Cooperate => "cooperate",
            Action::Defect => "defect",
        }
    }

    fn initial(self) -> char {
        match self {
            Action::Cooperate => 'C',
            Action::Defect => 'D',
        }
    }
}

/// L'interpretateur total des codes publics (act du module Lean).
#[allow(dead_code)]
fn act(agent: &BoundedAgent, opponent: ProgramCode) -> Action {
    match agent.code {
        ProgramCode::CooperateBot => Action::Cooperate,
        ProgramCode::DefectBot => Action::Defect,
        ProgramCode::Mirror => {
            if agent.budget == 0 {
                Action::Defect
            } else if opponent == ProgramCode::DefectBot {
                Action::Defect
            } else {
                Action::Cooperate
            }
        }
    }
}

/// Resultat ordonne de l'interpretateur : (action de row, action de column).
#[allow(dead_code)]
fn outcome_bounded(row: &BoundedAgent, column: &BoundedAgent) -> (Action, Action) {
    (act(row, column.code), act(column, row.code))
}

/// Le tuple Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_tuple(a: Action, b: Action) -> String {
    format!("('{}', '{}')", a.name(), b.name())
}

/// Le booleen Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_bool(v: bool) -> String {
    if v { "True".to_string() } else { "False".to_string() }
}

let cooperate_bot = BoundedAgent::new(ProgramCode::CooperateBot, 0);
let defect_bot_bounded = BoundedAgent::new(ProgramCode::DefectBot, 0);
let mirror_bot = BoundedAgent::new(ProgramCode::Mirror, 1);
let basic_family = vec![cooperate_bot, defect_bot_bounded, mirror_bot];

/// Organe de cooperation mutuelle (mutualCooperationCheck).
#[allow(dead_code)]
fn mutual_cooperation_check(left: &BoundedAgent, right: &BoundedAgent) -> bool {
    outcome_bounded(left, right) == (Action::Cooperate, Action::Cooperate)
}

/// Organe d'inexploitabilite sur une famille finie (unexploitableCheck).
#[allow(dead_code)]
fn unexploitable_check(agent: &BoundedAgent, opponents: &[BoundedAgent]) -> bool {
    opponents
        .iter()
        .all(|opp| outcome_bounded(agent, opp) != (Action::Cooperate, Action::Defect))
}

/// Organe d'equilibre relatif pour le PD canonique (programNashCheck).
#[allow(dead_code)]
fn program_nash_check(
    family: &[BoundedAgent],
    left: &BoundedAgent,
    right: &BoundedAgent,
) -> bool {
    let base = outcome_bounded(left, right);
    let base_left = payoff_rank(base.0, base.1);
    let base_right = payoff_rank(base.1, base.0);
    let left_ok = family.iter().all(|alt| {
        let o = outcome_bounded(alt, right);
        payoff_rank(o.0, o.1) <= base_left
    });
    let right_ok = family.iter().all(|alt| {
        let o = outcome_bounded(left, alt);
        payoff_rank(o.1, o.0) <= base_right
    });
    left_ok && right_ok
}

#[allow(dead_code)]
fn enregistrer(
    nom: &str,
    verdict: bool,
    theoreme_lean: &str,
    certificats: &mut Vec<(String, bool, String)>,
) {
    certificats.push((nom.to_string(), verdict, theoreme_lean.to_string()));
}

#[allow(dead_code)]
fn verifier(
    nom: &str,
    verdict: bool,
    theoreme_lean: &str,
    certificats: &mut Vec<(String, bool, String)>,
) {
    certificats.push((nom.to_string(), verdict, theoreme_lean.to_string()));
    println!("{:<42} Python={:<5}  <-  {}", nom, py_bool(verdict), theoreme_lean);
}

let mut certificats: Vec<(String, bool, String)> = Vec::new();

// Les deux premiers certificats sont enregistres sans impression : la
// cellule est autonome et reconstruit la liste complete.
enregistrer(
    "cooperate_cooperate",
    mutual_cooperation_check(&cooperate_bot, &cooperate_bot),
    "MutualCooperationBounded cooperateBot cooperateBot",
    &mut certificats,
);
enregistrer(
    "defect_defect",
    outcome_bounded(&defect_bot_bounded, &defect_bot_bounded) == (Action::Defect, Action::Defect),
    "outcomeBounded defectBotBounded defectBotBounded = (defect, defect)",
    &mut certificats,
);

// 3. Le bot miroir coopere avec lui-meme (budget positif)
verifier(
    "mirror_mirror",
    mutual_cooperation_check(&mirror_bot, &mirror_bot),
    "MutualCooperationBounded mirrorBot mirrorBot",
    &mut certificats,
);

// 4. Le bot defecteur est inexploitable contre toute famille :
//    son action n'est JAMAIS cooperate (argument structurel,
//    verifie ici sur la famille temoin)
verifier(
    "defectBotBounded_unexploitable",
    unexploitable_check(&defect_bot_bounded, &basic_family),
    "UnexploitableInFamily defectBotBounded opponents (quantifie)",
    &mut certificats,
);

// 5. Le miroir est inexploitable dans la famille temoin
verifier(
    "mirror_basicFamily_unexploitable",
    unexploitable_check(&mirror_bot, &basic_family),
    "unexploitableCheck mirrorBot basicFamily = true",
    &mut certificats,
);


mirror_mirror                              Python=True   <-  MutualCooperationBounded mirrorBot mirrorBot


defectBotBounded_unexploitable             Python=True   <-  UnexploitableInFamily defectBotBounded opponents (quantifie)


mirror_basicFamily_unexploitable           Python=True   <-  unexploitableCheck mirrorBot basicFamily = true


**Lecture chiffree — miroir, puis les deux inexploitabilites.** Trois lignes, trois natures. `mirror_mirror Python=True <- MutualCooperationBounded mirrorBot mirrorBot` : deux miroirs a budget positif cooperent entre eux — chacun inspecte une fois le code MIRROR et choisit coopération. `defectBotBounded_unexploitable Python=True <- UnexploitableInFamily defectBotBounded opponents` : le defecteur est inexploitable DANS la famille — le couple (cooperate, defect) ne se produit jamais pour lui, et l'énoncé Lean cité quantifie les listes d'opposants, tandis que ce carnet teste la famille témoin. `mirror_basicFamily_unexploitable Python=True <- unexploitableCheck mirrorBot basicFamily` : le miroir aussi, sur la meme famille temoin. Noter la difference d'echelle des trois preuves : la premiere se verifie sur une case, les deux autres exigent le balayage de la famille entiere.


### La proposition et son miroir decidable

Les certificats 6 et 7 présentent le même profil fixé sous deux formes : la proposition d’équilibre et la décision booléenne. Le module comporte aussi `programNashCheck_eq_true`, une équivalence générale supplémentaire entre la décision et la proposition ; elle ne transforme pas ces deux lignes calculées en preuves universelles.

Les deux formes ne sont pas redondantes : une proposition quantifiee se *prouve*, un organe booleen s'*execute*. Confronter l'une a l'autre est ce qui rend une divergence visible.

In [11]:
#[derive(Clone, Copy, PartialEq, Eq)]
enum ProgramCode {
    CooperateBot,
    DefectBot,
    Mirror,
}

impl ProgramCode {
    fn name(self) -> &'static str {
        match self {
            ProgramCode::CooperateBot => "COOPERATE_BOT",
            ProgramCode::DefectBot => "DEFECT_BOT",
            ProgramCode::Mirror => "MIRROR",
        }
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
struct BoundedAgent {
    code: ProgramCode,
    budget: i64,
}

impl BoundedAgent {
    fn new(code: ProgramCode, budget: i64) -> Self {
        BoundedAgent { code, budget }
    }
}

impl std::fmt::Display for BoundedAgent {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        write!(f, "BoundedAgent({}, budget={})", self.code.name(), self.budget)
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
enum Action {
    Cooperate,
    Defect,
}

impl Action {
    fn name(self) -> &'static str {
        match self {
            Action::Cooperate => "cooperate",
            Action::Defect => "defect",
        }
    }

    fn initial(self) -> char {
        match self {
            Action::Cooperate => 'C',
            Action::Defect => 'D',
        }
    }
}

/// L'interpretateur total des codes publics (act du module Lean).
#[allow(dead_code)]
fn act(agent: &BoundedAgent, opponent: ProgramCode) -> Action {
    match agent.code {
        ProgramCode::CooperateBot => Action::Cooperate,
        ProgramCode::DefectBot => Action::Defect,
        ProgramCode::Mirror => {
            if agent.budget == 0 {
                Action::Defect
            } else if opponent == ProgramCode::DefectBot {
                Action::Defect
            } else {
                Action::Cooperate
            }
        }
    }
}

/// Resultat ordonne de l'interpretateur : (action de row, action de column).
#[allow(dead_code)]
fn outcome_bounded(row: &BoundedAgent, column: &BoundedAgent) -> (Action, Action) {
    (act(row, column.code), act(column, row.code))
}

/// Le tuple Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_tuple(a: Action, b: Action) -> String {
    format!("('{}', '{}')", a.name(), b.name())
}

/// Le booleen Python, rendu comme Python le rend.
#[allow(dead_code)]
fn py_bool(v: bool) -> String {
    if v { "True".to_string() } else { "False".to_string() }
}

let cooperate_bot = BoundedAgent::new(ProgramCode::CooperateBot, 0);
let defect_bot_bounded = BoundedAgent::new(ProgramCode::DefectBot, 0);
let mirror_bot = BoundedAgent::new(ProgramCode::Mirror, 1);
let basic_family = vec![cooperate_bot, defect_bot_bounded, mirror_bot];

const PD_KEYS: [&str; 4] = ["T", "R", "P", "S"];
const PD_VALS: [i64; 4] = [5, 3, 1, 0];

#[allow(dead_code)]
fn pd(key: &str) -> i64 {
    for i in 0..PD_KEYS.len() {
        if PD_KEYS[i] == key {
            return PD_VALS[i];
        }
    }
    0
}

/// Paiement de l'agent ligne dans le PD canonique.
#[allow(dead_code)]
fn stage_payoff(row_action: Action, col_action: Action) -> i64 {
    if row_action == Action::Cooperate && col_action == Action::Cooperate {
        pd("R")
    } else if row_action == Action::Cooperate {
        pd("S")
    } else if col_action == Action::Cooperate {
        pd("T")
    } else {
        pd("P")
    }
}

/// Rang fini du module Lean : CC=3, CD=0, DC=5, DD=1.
#[allow(dead_code)]
fn payoff_rank(row_action: Action, col_action: Action) -> i64 {
    match (row_action, col_action) {
        (Action::Cooperate, Action::Cooperate) => 3,
        (Action::Cooperate, Action::Defect) => 0,
        (Action::Defect, Action::Cooperate) => 5,
        (Action::Defect, Action::Defect) => 1,
    }
}

/// Organe de cooperation mutuelle (mutualCooperationCheck).
#[allow(dead_code)]
fn mutual_cooperation_check(left: &BoundedAgent, right: &BoundedAgent) -> bool {
    outcome_bounded(left, right) == (Action::Cooperate, Action::Cooperate)
}

/// Organe d'inexploitabilite sur une famille finie (unexploitableCheck).
#[allow(dead_code)]
fn unexploitable_check(agent: &BoundedAgent, opponents: &[BoundedAgent]) -> bool {
    opponents
        .iter()
        .all(|opp| outcome_bounded(agent, opp) != (Action::Cooperate, Action::Defect))
}

/// Organe d'equilibre relatif pour le PD canonique (programNashCheck).
#[allow(dead_code)]
fn program_nash_check(
    family: &[BoundedAgent],
    left: &BoundedAgent,
    right: &BoundedAgent,
) -> bool {
    let base = outcome_bounded(left, right);
    let base_left = payoff_rank(base.0, base.1);
    let base_right = payoff_rank(base.1, base.0);
    let left_ok = family.iter().all(|alt| {
        let o = outcome_bounded(alt, right);
        payoff_rank(o.0, o.1) <= base_left
    });
    let right_ok = family.iter().all(|alt| {
        let o = outcome_bounded(left, alt);
        payoff_rank(o.1, o.0) <= base_right
    });
    left_ok && right_ok
}

#[allow(dead_code)]
fn enregistrer(
    nom: &str,
    verdict: bool,
    theoreme_lean: &str,
    certificats: &mut Vec<(String, bool, String)>,
) {
    certificats.push((nom.to_string(), verdict, theoreme_lean.to_string()));
}

/// Le nom textuel de l'action, comme la source.
#[allow(dead_code)]
fn action_of(nom: &str) -> Action {
    if nom == "cooperate" {
        Action::Cooperate
    } else {
        Action::Defect
    }
}

#[allow(dead_code)]
fn verifier(
    nom: &str,
    verdict: bool,
    theoreme_lean: &str,
    certificats: &mut Vec<(String, bool, String)>,
) {
    certificats.push((nom.to_string(), verdict, theoreme_lean.to_string()));
    println!("{:<42} Python={:<5}  <-  {}", nom, py_bool(verdict), theoreme_lean);
}

let mut certificats: Vec<(String, bool, String)> = Vec::new();

// La liste complete est reconstruite : les cinq premiers certificats sont
// enregistres sans impression, les trois derniers sont imprimes.
enregistrer(
    "cooperate_cooperate",
    mutual_cooperation_check(&cooperate_bot, &cooperate_bot),
    "MutualCooperationBounded cooperateBot cooperateBot",
    &mut certificats,
);
enregistrer(
    "defect_defect",
    outcome_bounded(&defect_bot_bounded, &defect_bot_bounded) == (Action::Defect, Action::Defect),
    "outcomeBounded defectBotBounded defectBotBounded = (defect, defect)",
    &mut certificats,
);
enregistrer(
    "mirror_mirror",
    mutual_cooperation_check(&mirror_bot, &mirror_bot),
    "MutualCooperationBounded mirrorBot mirrorBot",
    &mut certificats,
);
enregistrer(
    "defectBotBounded_unexploitable",
    unexploitable_check(&defect_bot_bounded, &basic_family),
    "UnexploitableInFamily defectBotBounded opponents (quantifie)",
    &mut certificats,
);
enregistrer(
    "mirror_basicFamily_unexploitable",
    unexploitable_check(&mirror_bot, &basic_family),
    "unexploitableCheck mirrorBot basicFamily = true",
    &mut certificats,
);

// 6. La defection mutuelle est un equilibre relatif dans la famille temoin
verifier(
    "defect_profile_programNash",
    program_nash_check(&basic_family, &defect_bot_bounded, &defect_bot_bounded),
    "ProgramNashBounded canonicalPD basicFamily defectBotBounded defectBotBounded",
    &mut certificats,
);

// 7. Le meme certificat via l'organe booleen (Lean : decide)
verifier(
    "defect_profile_check",
    program_nash_check(&basic_family, &defect_bot_bounded, &defect_bot_bounded),
    "programNashCheck basicFamily defectBotBounded defectBotBounded = true",
    &mut certificats,
);

// 8. Le rang fini preserve l'ordre des paiements (verification exhaustive 4x4)
let noms_actions = ["cooperate", "defect"];
let mut rang_ok = true;
for a1 in noms_actions.iter() {
    for a2 in noms_actions.iter() {
        for b1 in noms_actions.iter() {
            for b2 in noms_actions.iter() {
                let a = action_of(a1);
                let b = action_of(a2);
                let c = action_of(b1);
                let d = action_of(b2);
                if (payoff_rank(a, b) <= payoff_rank(c, d))
                    != (stage_payoff(a, b) <= stage_payoff(c, d))
                {
                    rang_ok = false;
                }
            }
        }
    }
}
verifier(
    "payoffRank_le_iff (16 paires)",
    rang_ok,
    "payoffRank_le_iff : rang fini iff ordre des paiements",
    &mut certificats,
);

let _conformes = certificats.iter().filter(|(_, v, _)| *v).count();
println!("\n{}/{} certificats conformes", _conformes, certificats.len());
assert!(
    certificats.iter().all(|(_, v, _)| *v),
    "divergence Rust/Lean detectee"
);


defect_profile_programNash                 Python=True   <-  ProgramNashBounded canonicalPD basicFamily defectBotBounded defectBotBounded


defect_profile_check                       Python=True   <-  programNashCheck basicFamily defectBotBounded defectBotBounded = true


payoffRank_le_iff (16 paires)              Python=True   <-  payoffRank_le_iff : rang fini iff ordre des paiements


8/8 certificats conformes


**Lecture chiffree — la ligne la plus riche : les 16 paires.** Apres les certificats 6 et 7 (l'equilibre relatif de la defection mutuelle, en double ecriture proposition/organe), la sortie affiche `payoffRank_le_iff (16 paires) Python=True <- payoffRank_le_iff : rang fini iff ordre des paiements`. Le compte 16 est la vraie information : toutes les paires de paiements du PD canonique (issues et rangs croises) sont passees en revue, et pour chacune l'equivalence a ete evaluee — le rang et l'ordre s'accordent paire par paire, sans exception. La ligne finale `8/8 certificats conformes` solde le rejeu : les huit vérifications annoncées ont chacune leur résultat calculé en Rust, le partage des roles reste ce que la section suivante en dit.


Les 8 verdicts calcules par ce carnet Rust sont conformes aux cas calcules. Le module Lean cité comprend des cas fixés, un énoncé sur les listes d’opposants et, séparément, une équivalence générale pour `programNashCheck` ; ce carnet n’exécute aucune preuve Lean.


## Exercice 3 - Famille etendue aux budgets

On etend la famille temoin avec des variantes de budget : `basic_family_etendue = basic_family + [BoundedAgent(ProgramCode.MIRROR, 5), mirror_broke]`. **Question** : le profil de defection mutuelle `(defect_bot_bounded, defect_bot_bounded)` reste-t-il un equilibre relatif dans cette famille etendue ? Et la cooperation mutuelle des miroirs `(mirror_bot, mirror_bot)` devient-elle equilibre ? Completer le calcul des deux verdicts et interpreter pourquoi la reponse differe entre les deux profils.


In [12]:
#[derive(Clone, Copy, PartialEq, Eq)]
enum ProgramCode {
    CooperateBot,
    DefectBot,
    Mirror,
}

impl ProgramCode {
    fn name(self) -> &'static str {
        match self {
            ProgramCode::CooperateBot => "COOPERATE_BOT",
            ProgramCode::DefectBot => "DEFECT_BOT",
            ProgramCode::Mirror => "MIRROR",
        }
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
struct BoundedAgent {
    code: ProgramCode,
    budget: i64,
}

impl BoundedAgent {
    fn new(code: ProgramCode, budget: i64) -> Self {
        BoundedAgent { code, budget }
    }
}

impl std::fmt::Display for BoundedAgent {
    fn fmt(&self, f: &mut std::fmt::Formatter<'_>) -> std::fmt::Result {
        write!(f, "BoundedAgent({}, budget={})", self.code.name(), self.budget)
    }
}

#[derive(Clone, Copy, PartialEq, Eq)]
enum Action {
    Cooperate,
    Defect,
}

impl Action {
    fn name(self) -> &'static str {
        match self {
            Action::Cooperate => "cooperate",
            Action::Defect => "defect",
        }
    }

    fn initial(self) -> char {
        match self {
            Action::Cooperate => 'C',
            Action::Defect => 'D',
        }
    }
}

/// L'interpretateur total des codes publics (act du module Lean).
#[allow(dead_code)]
fn act(agent: &BoundedAgent, opponent: ProgramCode) -> Action {
    match agent.code {
        ProgramCode::CooperateBot => Action::Cooperate,
        ProgramCode::DefectBot => Action::Defect,
        ProgramCode::Mirror => {
            if agent.budget == 0 {
                Action::Defect
            } else if opponent == ProgramCode::DefectBot {
                Action::Defect
            } else {
                Action::Cooperate
            }
        }
    }
}

/// Resultat ordonne de l'interpretateur : (action de row, action de column).
#[allow(dead_code)]
fn outcome_bounded(row: &BoundedAgent, column: &BoundedAgent) -> (Action, Action) {
    (act(row, column.code), act(column, row.code))
}

let cooperate_bot = BoundedAgent::new(ProgramCode::CooperateBot, 0);
let defect_bot_bounded = BoundedAgent::new(ProgramCode::DefectBot, 0);
let mirror_bot = BoundedAgent::new(ProgramCode::Mirror, 1);
let basic_family = vec![cooperate_bot, defect_bot_bounded, mirror_bot];

const PD_KEYS: [&str; 4] = ["T", "R", "P", "S"];
const PD_VALS: [i64; 4] = [5, 3, 1, 0];

#[allow(dead_code)]
fn pd(key: &str) -> i64 {
    for i in 0..PD_KEYS.len() {
        if PD_KEYS[i] == key {
            return PD_VALS[i];
        }
    }
    0
}

/// Paiement de l'agent ligne dans le PD canonique.
#[allow(dead_code)]
fn stage_payoff(row_action: Action, col_action: Action) -> i64 {
    if row_action == Action::Cooperate && col_action == Action::Cooperate {
        pd("R")
    } else if row_action == Action::Cooperate {
        pd("S")
    } else if col_action == Action::Cooperate {
        pd("T")
    } else {
        pd("P")
    }
}

/// Rang fini du module Lean : CC=3, CD=0, DC=5, DD=1.
#[allow(dead_code)]
fn payoff_rank(row_action: Action, col_action: Action) -> i64 {
    match (row_action, col_action) {
        (Action::Cooperate, Action::Cooperate) => 3,
        (Action::Cooperate, Action::Defect) => 0,
        (Action::Defect, Action::Cooperate) => 5,
        (Action::Defect, Action::Defect) => 1,
    }
}

/// Organe de cooperation mutuelle (mutualCooperationCheck).
#[allow(dead_code)]
fn mutual_cooperation_check(left: &BoundedAgent, right: &BoundedAgent) -> bool {
    outcome_bounded(left, right) == (Action::Cooperate, Action::Cooperate)
}

/// Organe d'inexploitabilite sur une famille finie (unexploitableCheck).
#[allow(dead_code)]
fn unexploitable_check(agent: &BoundedAgent, opponents: &[BoundedAgent]) -> bool {
    opponents
        .iter()
        .all(|opp| outcome_bounded(agent, opp) != (Action::Cooperate, Action::Defect))
}

/// Organe d'equilibre relatif pour le PD canonique (programNashCheck).
#[allow(dead_code)]
fn program_nash_check(
    family: &[BoundedAgent],
    left: &BoundedAgent,
    right: &BoundedAgent,
) -> bool {
    let base = outcome_bounded(left, right);
    let base_left = payoff_rank(base.0, base.1);
    let base_right = payoff_rank(base.1, base.0);
    let left_ok = family.iter().all(|alt| {
        let o = outcome_bounded(alt, right);
        payoff_rank(o.0, o.1) <= base_left
    });
    let right_ok = family.iter().all(|alt| {
        let o = outcome_bounded(left, alt);
        payoff_rank(o.1, o.0) <= base_right
    });
    left_ok && right_ok
}

let mirror_broke = BoundedAgent::new(ProgramCode::Mirror, 0);

let basic_family_etendue: Vec<BoundedAgent> = {
    let mut famille = basic_family.clone();
    famille.push(BoundedAgent::new(ProgramCode::Mirror, 5));
    famille.push(mirror_broke);
    famille
};

/// Renvoie (nash_defection, nash_cooperation_miroirs) dans la famille etendue.
/// TODO Etudiant : appeler program_nash_check deux fois sur
/// basic_family_etendue avec les profils (defect, defect) puis
/// (mirror_bot, mirror_bot).
#[allow(dead_code)]
fn equilibres_famille_etendue(
    _famille: &[BoundedAgent],
    _def_bot: &BoundedAgent,
    _mir: &BoundedAgent,
) -> Option<(bool, bool)> {
    None // TODO etudiant
}

println!("Exercice 3 a completer : equilibres_famille_etendue()");
println!("Indice : contre un miroir a budget positif, que rapporte la deviation ?");

// Plusieurs familles distinguent un calcul general d'une reponse memorisee.
let _cas3: Vec<(Vec<BoundedAgent>, BoundedAgent, BoundedAgent)> = vec![
    (basic_family_etendue.clone(), defect_bot_bounded, mirror_bot),
    (vec![cooperate_bot], cooperate_bot, cooperate_bot),
    (vec![defect_bot_bounded], defect_bot_bounded, defect_bot_bounded),
];
let _attendus3: Vec<(bool, bool)> = _cas3
    .iter()
    .map(|(f, d, m)| (program_nash_check(f, d, d), program_nash_check(f, m, m)))
    .collect();
let _reps3: Vec<Option<(bool, bool)>> = _cas3
    .iter()
    .map(|(f, d, m)| equilibres_famille_etendue(f, d, m))
    .collect();
assert!(
    _reps3.iter().all(|r| r.is_none()) || _reps3.iter().all(|r| r.is_some()),
    "Reponse partielle"
);
if _reps3[0].is_some() {
    assert!(
        _reps3
            .iter()
            .zip(_attendus3.iter())
            .all(|(r, e)| r.as_ref() == Some(e)),
        "Equilibres incorrects"
    );
};


Exercice 3 a completer : equilibres_famille_etendue()


Indice : contre un miroir a budget positif, que rapporte la deviation ?


## À retenir

- **Le budget est public et paie la lecture** : la famille témoin vit à budget **0** pour le coopérateur et le défecteur — leur décision ne regarde ni l'adversaire ni l'historique — et le miroir seul porte **1** ; ce seul entier sépare le miroir qui réfléchit du miroir cassé.
- **La fonction `act` est totale** : aucune recherche de preuve, seulement l'inspection du code adverse et du budget ; à budget nul le miroir dévie, à budget positif il coopère sauf contre le code explicitement défecteur — et deux miroirs produisent une fois `(cooperate, cooperate)`.
- **Un rang fini pour l'ordre du dilemme** : sur le PD canonique (`T=5, R=3, P=1, S=0`), `payoffRank` aplatit l'ordre en **3** pour (C, C), **0** pour (C, D), **5** pour (D, C), **1** pour (D, D) — ces valeurs reprennent explicitement les gains canoniques et préservent leur ordre, sans établir de comparaison cardinale entre jeux.
- **Rejeu intégral des certificats** : le vérificateur Rust indépendant reproduit les huit cas annoncés — **8/8** conformes, dont l'équivalence du rang fini vérifiée paire par paire sur **16** croisements.
- **Le partage des rôles** : ce carnet (Rust) vérifie une matrice finie ; les énoncés Lean cités ont chacun leur portée propre, dont la quantification sur les listes d’opposants et l’équivalence générale séparée. Le carnet ne les exécute pas.

## Vérifiez votre compréhension

1. Quel budget le miroir porte-t-il dans la famille témoin, et que fait `act` pour un miroir à budget nul ?
   <details><summary>Réponse</summary>

   **Réponse.** Le miroir porte `budget=1` — un budget positif permet le test du code public. À budget nul, `act` renvoie `defect` immédiatement, sans inspection : la règle à budget nul prescrit immédiatement la défection.

   </details>

2. Quels rangs `payoffRank` assigne-t-il aux quatre issues, et quel ordre du dilemme ce classement encode-t-il ?
   <details><summary>Réponse</summary>

   **Réponse.** **3** pour (cooperate, cooperate), **0** pour (cooperate, defect), **5** pour (defect, cooperate), **1** pour (defect, defect) : l'ordre total T > R > P > S du PD canonique (`T=5, R=3, P=1, S=0`), aplati sur des entiers.

   </details>

3. Que dit la ligne finale `8/8 certificats conformes`, et que ne dit-elle pas ?
   <details><summary>Réponse</summary>

   **Réponse.** Que les huit cas annoncés ont le résultat attendu dans ce carnet Rust, dont `payoffRank_le_iff` sur les **16** paires d’issues de `canonicalPD`. Ils ne démontrent pas des affirmations sur toutes les familles ou tous les budgets ; le module cité comporte notamment un énoncé sur les listes d’opposants et une équivalence générale distincte.

   </details>

4. Contre quel membre de la famille le coopérateur subit-il `(cooperate, defect)`, et pourquoi ne s'adapte-t-il pas ?
   <details><summary>Réponse</summary>

   **Réponse.** Contre `DEFECT_BOT b=0` : la sortie rend `('cooperate', 'defect')` pour cette paire. Le coopérateur coopère sans exception ni mémoire — il subit, il ne s'adapte pas.

   </details>

## Pour aller plus loin

**Carnet suivant.** [Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python](Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python.ipynb) - le coût du raisonnement borné, prolongement direct du budget explicite.

**Carnets voisins.** Pivot conceptuel : [Dynamics-04-Open-Source-Game-Theory-Python.ipynb](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) ; compagnon Lean : [Dynamics-05-Agents-Bornes-Lean.ipynb](Dynamics-05-Agents-Bornes-Lean.ipynb) ; module Lean (FR) : [game_theory_lean/ProgramGames/Bounded.lean](../game_theory_lean/ProgramGames/Bounded.lean) - miroir anglais : [Bounded_en.lean](../game_theory_lean/ProgramGames/Bounded_en.lean) ; noyau fonctionnel : [game_theory_lean/ProgramGames/Basic.lean](../game_theory_lean/ProgramGames/Basic.lean) ; lake complet : [game_theory_lean/README.md](../game_theory_lean/README.md).

**Lectures.** Barasz et al. (2014) et Critch (2016) - le modèle structurel de l'agent-programme à code public et budget de raisonnement fini que ce compagnon reproduit indépendamment depuis zéro.

## Navigation et suite

- Pivot conceptuel : [`Dynamics-04-Open-Source-Game-Theory-Python.ipynb`](Dynamics-04-Open-Source-Game-Theory-Python.ipynb)
- Module Lean (FR) : [`game_theory_lean/ProgramGames/Bounded.lean`](../game_theory_lean/ProgramGames/Bounded.lean) - miroir anglais : [`Bounded_en.lean`](../game_theory_lean/ProgramGames/Bounded_en.lean)
- Noyau fonctionnel : [`game_theory_lean/ProgramGames/Basic.lean`](../game_theory_lean/ProgramGames/Basic.lean)
- Lake complet : [`game_theory_lean/README.md`](../game_theory_lean/README.md)

Ce compagnon couvre le versant **calcul fini** du modele borne ; le versant formel vit dans le lake : les huit lignes ci-dessus ne sont pas huit théorèmes universels. Les exercices 1 a 3 restent a completer.
